# 🧠 BiLSTM + GRU — Predicción del número semanal de tweets

Este notebook combina una **capa LSTM bidireccional** con una capa **GRU** para capturar dependencias temporales en ambos sentidos.

La bidireccionalidad permite que la red procese la secuencia tanto hacia adelante como hacia atrás, obteniendo representaciones más ricas de la tendencia y la estacionalidad.

**Estructura:**
1. Setup y carga de datos
2. Preprocesamiento y ventanas deslizantes
3. Train / Val / Test split
4. Carga del modelo desde `models/lstm_gru_model.py`
5. Entrenamiento con curvas de convergencia
6. Evaluación en los tres splits
7. Resumen de resultados


In [ ]:
import os
if not os.path.exists('AP'):
    !git clone https://github.com/0xnito/AP.git
%cd AP
!pip install -q kaggle tensorflow matplotlib scikit-learn

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print('TensorFlow:', tf.__version__)
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
os.makedirs('data/raw', exist_ok=True)
csv_path = 'data/raw/all_musk_posts.csv'

if os.path.exists(csv_path):
    print('✅ Dataset disponible.')
elif os.path.exists('/root/.kaggle/kaggle.json'):
    !kaggle datasets download -d dadalyndell/elon-musk-tweets-2010-to-2025-march -p data/raw
    !unzip -o data/raw/elon-musk-tweets-2010-to-2025-march.zip -d data/raw
else:
    print('⚠️  Sube kaggle.json o descarga el CSV manualmente.')

df = pd.read_csv(csv_path)
df['createdAt'] = pd.to_datetime(df['createdAt'], utc=True)
df['createdAt_naive'] = df['createdAt'].dt.tz_localize(None)
df['week'] = df['createdAt_naive'].dt.to_period('W').apply(lambda r: r.start_time)

weekly = (
    df.groupby('week').size()
      .reset_index(name='tweet_count')
      .sort_values('week')
      .reset_index(drop=True)
)
print(f'Semanas: {len(weekly)}')

In [ ]:
SEQ_LEN = 8
BATCH_SIZE = 32
EPOCHS = 150
PATIENCE = 20

values = weekly['tweet_count'].values.astype(np.float32).reshape(-1, 1)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(values).flatten()

def make_windows(series, seq_len):
    X, y = [], []
    for i in range(len(series) - seq_len):
        X.append(series[i:i + seq_len])
        y.append(series[i + seq_len])
    return np.array(X)[..., np.newaxis], np.array(y)

X_all, y_all = make_windows(scaled, SEQ_LEN)
n = len(X_all)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

X_train, y_train = X_all[:n_train],               y_all[:n_train]
X_val,   y_val   = X_all[n_train:n_train+n_val],  y_all[n_train:n_train+n_val]
X_test,  y_test  = X_all[n_train+n_val:],         y_all[n_train+n_val:]

print(f'Train: {len(X_train):>4} | Val: {len(X_val):>4} | Test: {len(X_test):>4}')

## 4. Modelo BiLSTM + GRU

In [ ]:
from models.lstm_gru_model import build_lstm_gru_model

model = build_lstm_gru_model(
    seq_len=SEQ_LEN,
    n_features=1,
    units=64,
    dropout=0.2,
    learning_rate=1e-3,
)
model.summary()
print(f'\n🔢 Parámetros totales: {model.count_params():,}')

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=10,
        min_lr=1e-6, verbose=1
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Pérdida (MSE) — BiLSTM+GRU', fontsize=13)
axes[0].set_xlabel('Época'); axes[0].set_ylabel('MSE')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'],     label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('MAE — BiLSTM+GRU', fontsize=13)
axes[1].set_xlabel('Época'); axes[1].set_ylabel('MAE (escalado)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Curvas de entrenamiento — Modelo BiLSTM+GRU', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('data/raw/bilstm_gru_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Épocas: {len(history.history["loss"])}')

In [ ]:
def evaluate_split(model, X, y_true_scaled, scaler, split_name):
    y_pred_scaled = model.predict(X, verbose=0).flatten()
    y_true = scaler.inverse_transform(y_true_scaled.reshape(-1, 1)).flatten()
    y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    print(f'  {split_name:<12}  RMSE={rmse:7.2f}  MAE={mae:7.2f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    return dict(split=split_name, RMSE=rmse, MAE=mae, R2=r2, MAPE=mape,
                y_true=y_true, y_pred=y_pred)

print('\n📊 Resultados BiLSTM+GRU (escala original):')
print('─' * 65)
res_train = evaluate_split(model, X_train, y_train, scaler, 'Train')
res_val   = evaluate_split(model, X_val,   y_val,   scaler, 'Validación')
res_test  = evaluate_split(model, X_test,  y_test,  scaler, 'Test')
print('─' * 65)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

axes[0].plot(res_test['y_true'], label='Real')
axes[0].plot(res_test['y_pred'], label='Predicción BiLSTM+GRU', alpha=0.85)
axes[0].set_title('Test — Real vs Predicción BiLSTM+GRU', fontsize=13)
axes[0].set_xlabel('Índice de semana (test)'); axes[0].set_ylabel('Nº tweets')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].scatter(res_test['y_true'], res_test['y_pred'],
                alpha=0.6, edgecolors='k', linewidths=0.3, s=50)
lims = [min(res_test['y_true'].min(), res_test['y_pred'].min()),
        max(res_test['y_true'].max(), res_test['y_pred'].max())]
axes[1].plot(lims, lims, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[1].set_title(f'Scatter Test (R²={res_test["R2"]:.4f})', fontsize=13)
axes[1].set_xlabel('Real'); axes[1].set_ylabel('Predicho')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/raw/bilstm_gru_predictions.png', dpi=150)
plt.show()

In [ ]:
results_df = pd.DataFrame([
    {'Split': r['split'], 'RMSE': round(r['RMSE'], 2), 'MAE': round(r['MAE'], 2),
     'R²': round(r['R2'], 4), 'MAPE (%)': round(r['MAPE'], 2)}
    for r in [res_train, res_val, res_test]
])
print('\n📋 Resumen de métricas — Modelo BiLSTM+GRU')
print('='*55)
print(results_df.to_string(index=False))
print('='*55)
print(f'Parámetros totales: {model.count_params():,}')

os.makedirs('saved_models', exist_ok=True)
model.save('saved_models/bilstm_gru_tweet_predictor.keras')
print('\n✅ Modelo guardado.')